# 3. Data processing and feature engineering

Aquí definimos las transformaciones, pero **no ajustamos un scaler sobre todo el conjunto de entrenamiento**. Los preprocesadores se incluirán dentro de cada pipeline para que se ajusten nuevamente en cada fold de validación cruzada.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd "/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification"

/content/drive/MyDrive/Courses/SI3003 - Inteligencia Artificial/Clase06/Classification


In [3]:
import json
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [7]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"
for directory in (DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

In [8]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
target = contract["target"]
feature_columns = contract["features"]

assert set(feature_columns) == set(train_df.columns) - {target}
print(f"Contrato validado para {len(feature_columns)} variables.")

Contrato validado para 30 variables.


## Dos estrategias de preprocesamiento

In [9]:
numeric_for_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

tree_preprocessor = ColumnTransformer([
    ("numeric", numeric_for_tree, feature_columns),
], verbose_feature_names_out=False)

scaled_preprocessor = ColumnTransformer([
    ("numeric", numeric_scaled, feature_columns),
], verbose_feature_names_out=False)

preprocessors = {
    "tree": tree_preprocessor,
    "scaled": scaled_preprocessor,
}
joblib.dump(preprocessors, ARTIFACTS_DIR / "preprocessors.joblib")
preprocessors

{'tree': ColumnTransformer(transformers=[('numeric',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='median'))]),
                                  ['mean radius', 'mean texture',
                                   'mean perimeter', 'mean area',
                                   'mean smoothness', 'mean compactness',
                                   'mean concavity', 'mean concave points',
                                   'mean symmetry', 'mean fractal dimension',
                                   'radius error', 'texture error',
                                   'perimeter error', 'area error',
                                   'smoothness error', 'compactness error',
                                   'concavity error', 'concave points error',
                                   'symmetry error', 'fractal dimension error',
                                   'worst radius', 'worst texture',

## ¿Dónde está el feature engineering?

En este dataset las 30 variables ya fueron calculadas a partir de imágenes digitalizadas. Por eso no inventamos nuevas variables sin una hipótesis de dominio. El pipeline queda preparado para añadir transformadores en el futuro.

La salida de este notebook son **definiciones no ajustadas**. El ajuste ocurrirá dentro de `GridSearchCV` en el notebook siguiente.